In [ ]:
import geoengine as ge
import geoengine_api_client as geoc


In [ ]:
ge.initialize("http://localhost:3030/api", ("admin@localhost", "adminadmin"))

In [ ]:
# a tiny static raster from the `test_data` volume is enough for this test
file_path = "raster/landcover/landcover.tif"  # relative to the `test_data` volume

gdal_params = geoc.GdalDatasetParameters.from_dict({
    "filePath": file_path,
    "rasterbandChannel": 1,
    "geoTransform": {"originCoordinate": {"x": -180.0, "y": 90.0}, "xPixelSize": 0.1, "yPixelSize": -0.1},
    "width": 3600,
    "height": 1800,
    "fileNotFoundHandling": "NoData",
    "allowAlphabandAsMask": True,
})

measurement = ge.ClassificationMeasurement(measurement="Land Cover", classes={0: "Water", 1: "Land"})

result_descriptor = ge.RasterResultDescriptor(
    "U8",
    [ge.RasterBandDescriptor("band", measurement)],
    "EPSG:4326",
    spatial_grid=ge.SpatialGridDescriptor(
        descriptor=geoc.SpatialGridDescriptorState.SOURCE,
        spatial_grid=ge.SpatialGridDefinition(
            geo_transform=ge.GeoTransform(x_min=-180.0, y_max=90.0, x_pixel_size=0.1, y_pixel_size=-0.1),
            grid_bounds=ge.GridBoundingBox2D(
                top_left_idx=ge.GridIdx2D(x_idx=0, y_idx=0), bottom_right_idx=ge.GridIdx2D(x_idx=3599, y_idx=1799)
            ),
        ),
    ),
    time=ge.TimeDescriptor(dimension=ge.IrregularTimeDimension(), bounds=None),
)

meta_data = geoc.MetaDataDefinition(geoc.GdalMetaDataStatic.from_dict({
    "type": "GdalStatic",
    "time": None,
    "params": gdal_params,
    "resultDescriptor": result_descriptor.to_api_dict().to_dict(),
}))


In [ ]:
properties = ge.datasets.AddDatasetProperties(
    name="OVERWRITE_TEST_DATASET",
    display_name="Overwrite Test Dataset",
    description="Small land cover raster used to test the overwrite functions.",
    source_operator="GdalSource",
)

volume = ge.volume_by_name("test_data")


In [ ]:
# register the dataset for the first time
dataset_name = ge.add_or_replace_dataset_with_permissions(
    volume,
    properties,
    meta_data,
    replace_existing=True,
)
dataset_id_before = ge.datasets.dataset_info_by_name(dataset_name).id
print("created:", dataset_name, "id:", dataset_id_before)


In [ ]:
# re-register with replace_existing=True -> the dataset must be deleted and recreated
dataset_name = ge.add_or_replace_dataset_with_permissions(
    volume,
    properties,
    meta_data,
    replace_existing=True,
)
dataset_id_after = ge.datasets.dataset_info_by_name(dataset_name).id
print("replaced:", dataset_name, "id:", dataset_id_after)

assert dataset_id_after != dataset_id_before, "dataset was not replaced (id unchanged)"


In [ ]:
# re-register with replace_existing=False -> the existing dataset must be kept
dataset_name = ge.add_or_replace_dataset_with_permissions(
    volume,
    properties,
    meta_data,
    replace_existing=False,
)
dataset_id_kept = ge.datasets.dataset_info_by_name(dataset_name).id
print("kept:", dataset_name, "id:", dataset_id_kept)

assert dataset_id_kept == dataset_id_after, "dataset was replaced although replace_existing=False"


In [ ]:
# a dedicated test collection, (re)created from scratch for each run
root_of_layerdb = ge.layer_collection().items[0].load()
test_collection = root_of_layerdb.get_or_create_unique_collection(
    "Overwrite Test",
    "Overwrite Test",
    delete_existing_with_same_name=True,
)


In [ ]:
layer_name = "Overwrite Test Layer"

layer_id_before = test_collection.add_layer_with_permissions(
    name=layer_name,
    description="test",
    workflow=ge.workflow_builder.operators.GdalSource(dataset_name),
    symbology=None,
    replace_existing=True,
)
print("added:", layer_name, "id:", layer_id_before)


In [ ]:
# re-add with replace_existing=True -> the old layer must be removed first
layer_id_after = test_collection.add_layer_with_permissions(
    name=layer_name,
    description="test",
    workflow=ge.workflow_builder.operators.GdalSource(dataset_name),
    symbology=None,
    replace_existing=True,
)
print("replaced:", layer_name, "id:", layer_id_after)

remaining = test_collection.get_items_by_name(layer_name)
print("layers with name", layer_name, ":", len(remaining))

assert layer_id_after != layer_id_before, "layer was not replaced (id unchanged)"
assert len(remaining) == 1, "expected exactly one layer with this name"
assert remaining[0].listing_id == layer_id_after, "the remaining layer must be the new one"


In [ ]:
# remove the test collection and dataset again
test_collection.remove()
ge.datasets.delete_dataset(dataset_name)
print("cleaned up test data")
